In [1]:
import pandas as pd
df = pd.read_csv("data_cleaning_finale.csv", sep=",")  
df

,content
0,rt doreencaven fear people watch many performa...
1,rt karoromitchelle spend rest trapped sneaky l...
2,rt urgent appeal action express much feel man ...
3,rt shivanyasitole supreme god kabir god kabir ...
4,rt trust im leaving staying staying always wis...
...,...
12301,catasters
12302,zegalbamount fuji seen international space sta...
12303,thehopefulquotes loving someone mean taking ri...
12304,perfectfeelings want talk happened without men...


In [2]:
from textblob import TextBlob

In [3]:

# Vérifier et convertir la colonne "content" en texte, gérer les valeurs manquantes
df["content"] = df["content"].astype(str).fillna("")

# Fonction pour attribuer un label (1 = déprimé, 0 = non déprimé)
def assign_label(text):
    sentiment_score = TextBlob(text).sentiment.polarity
    return 1 if sentiment_score < 0 else 0  

# Appliquer l'analyse des sentiments
df["label"] = df["content"].apply(assign_label)

# Sauvegarder le fichier mis à jour
df.to_csv("data_with_labels.csv", index=False)

# Afficher un aperçu des données mises à jour
print(df.head())


                                             content  label
0  rt doreencaven fear people watch many performa...      0
1  rt karoromitchelle spend rest trapped sneaky l...      1
2  rt urgent appeal action express much feel man ...      0
3  rt shivanyasitole supreme god kabir god kabir ...      1
4  rt trust im leaving staying staying always wis...      0


In [4]:
# 1. Charger les données
data = pd.read_csv("data_with_labels.csv")

In [5]:
data.isnull().sum()

content    7
label      0
dtype: int64

In [6]:
data = data.dropna(subset=["content"])

In [7]:
num_duplicates = data.duplicated().sum()
print(f"Le nombre de lignes dupliquées est : {num_duplicates}")

Le nombre de lignes dupliquées est : 288


In [8]:
# Supprimer les lignes dupliquées
data = data.drop_duplicates()

In [9]:
print(data['label'].value_counts())


label
0    7572
1    4439
Name: count, dtype: int64


In [10]:
print(data.sample(10)[["content", "label"]])
#  les labels sont bruités ou mal attribués


                                                content  label
6066  samaaldias chandaleew one need break heart lis...      0
4245  carnivorenewb reach point actually feel hungry...      0
7668  day feel lonely day feel like dont give single...      0
2880  rt superstarhoshi beautiful moment raw unplann...      0
7804  coldplay hear keep grinding homie also keep go...      0
9232                               idrisi na mood swing      0
3876  rt subhatchman yes make sense due severe pain ...      1
566   struggle may tough remember one day become str...      1
1367  began learning consumer law finding increased ...      0
9094  rt paigeota teenager understand mood swing tee...      0


In [11]:
data["content"] = data["content"].str.replace(r"^rt\s+", "", regex=True)


In [12]:
#  Préparer les données
X = df["content"]  
y = df["label"]  

In [13]:
#  Transformer le texte en vecteurs TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_vectorized = vectorizer.fit_transform(X)

In [14]:
from imblearn.over_sampling import SMOTE

# Appliquer SMOTE
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_vectorized, y)

In [15]:
# . Séparer les données en train et test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)

In [16]:
# Sélectionner les 10 premiers échantillons pour le test
X_final_test = X_test[:10].toarray()  
y_final_test = y_test[:10].values  

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

#  Entraîner le modèle
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

# Évaluer le modèle
y_pred = model_rf.predict(X_test)

# Afficher la précision
accuracy = accuracy_score(y_test, y_pred)
print(f"Précision du modèle : {accuracy:.2f}")

# Afficher le rapport de classification
print("\nRapport de classification :\n")
print(classification_report(y_test, y_pred))


Précision du modèle : 0.85

Rapport de classification :

              precision    recall  f1-score   support

           0       0.86      0.92      0.89      1568
           1       0.84      0.74      0.78       894

    accuracy                           0.85      2462
   macro avg       0.85      0.83      0.84      2462
weighted avg       0.85      0.85      0.85      2462



In [18]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Entraîner le modèle SVM
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train, y_train)

# Prédictions et évaluation
y_pred = svm_model.predict(X_test)

# Afficher la précision et le rapport
accuracy = accuracy_score(y_test, y_pred)
print(f"Précision du modèle : {accuracy:.2f}")
print("\nRapport de classification :\n")
print(classification_report(y_test, y_pred))


Précision du modèle : 0.87

Rapport de classification :

              precision    recall  f1-score   support

           0       0.87      0.93      0.90      1568
           1       0.87      0.75      0.81       894

    accuracy                           0.87      2462
   macro avg       0.87      0.84      0.85      2462
weighted avg       0.87      0.87      0.87      2462



In [19]:
from sklearn.linear_model import LogisticRegression


# Entraîner le modèle
l_model = LogisticRegression(max_iter=1000, random_state=42)
l_model.fit(X_train, y_train)

# Prédictions et évaluation
y_pred = l_model.predict(X_test)

# Afficher la précision et le rapport
accuracy = accuracy_score(y_test, y_pred)
print(f"Précision du modèle : {accuracy:.2f}")
print("\nRapport de classification :\n")
print(classification_report(y_test, y_pred))


Précision du modèle : 0.83

Rapport de classification :

              precision    recall  f1-score   support

           0       0.82      0.95      0.88      1568
           1       0.88      0.62      0.73       894

    accuracy                           0.83      2462
   macro avg       0.85      0.79      0.80      2462
weighted avg       0.84      0.83      0.82      2462



In [20]:
# import tensorflow as tf
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense
# from sklearn.metrics import accuracy_score, classification_report

# # Construire le modèle ANN simple
# ann_model = Sequential([
#     Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
#     Dense(64, activation='relu'),
#     Dense(1, activation='sigmoid')  # Sigmoid pour classification binaire
# ])

# # Compiler le modèle
# ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# # Entraîner le modèle avec validation accuracy
# history = ann_model.fit(X_train.toarray(), y_train, 
#                         epochs=10, batch_size=32, 
#                         validation_data=(X_test.toarray(), y_test),
#                         verbose=1)

# # Prédictions et évaluation
# y_pred_probs = ann_model.predict(X_test.toarray())
# y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# # Afficher la précision et le rapport
# accuracy = accuracy_score(y_test, y_pred)
# print(f"Précision du modèle : {accuracy:.2f}")
# print("\nRapport de classification :\n")
# print(classification_report(y_test, y_pred))




In [30]:
# Example test
def predict_sentiment(text):
    text_vectorized = vectorizer.transform([text])
    prediction =svm_model.predict(text_vectorized)[0]
    return "Depressed" if prediction == 1 else "Not Depressed"

example_text = ""
print(f"Text: {example_text} | Prediction: {predict_sentiment(example_text)}")


Text: i dont wanna go outside today | Prediction: Not Depressed


In [22]:
import pickle
# Sauvegarder le modèle Random Forest
with open('svm_depression_model.pkl', 'wb') as file:
    pickle.dump(svm_model, file)



In [23]:
import pickle

# Charger le modèle sauvegardé
with open("svm_depression_model.pkl", "rb") as file:
    svm_model = pickle.load(file)

# Sélectionner les 10 premiers échantillons pour le test
X_final_test = X_test[:10].toarray()  # Convertir en array si nécessaire
y_final_test = y_test[:10].values  # Récupérer les vraies étiquettes

# Vérification de la forme des données de test
print(f"Shape de X_final_test : {X_final_test.shape}")

# Faire des prédictions sur ces 10 échantillons
y_pred = svm_model.predict(X_final_test)

# Affichage des résultats
print(f"Prédictions du modèle : {y_pred}")
print(f"Valeurs réelles : {y_final_test}")


Shape de X_final_test : (10, 26585)
Prédictions du modèle : [1 0 0 0 0 1 0 0 1 0]
Valeurs réelles : [1 0 0 0 0 1 1 0 1 0]


In [24]:
# import pandas as pd
# from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# # Création de l'analyseur de sentiments
# analyzer = SentimentIntensityAnalyzer()

# # Vérifier et convertir la colonne "content" en texte, gérer les valeurs manquantes
# df["content"] = df["content"].astype(str).fillna("")

# # Fonction pour analyser le sentiment et attribuer un label
# def assign_label(text):
#     score = analyzer.polarity_scores(text)["compound"]
#     if score >= 0.05:
#         return "positif"  # 😊
#     elif score <= -0.05:
#         return "négatif"  # 😞
#     else:
#         return "neutre"  # 😐

# # Appliquer l'analyse des sentiments sur la colonne "content"
# df["sentiment"] = df["content"].apply(assign_label)

# # Sauvegarder le fichier mis à jour
# df.to_csv("data_with_sentiments.csv", index=False)

# # Afficher un aperçu des données mises à jour
# print(df.head())
